# Baseline — Predicting Molecular Scalar Couplings

**Competition:** predict the **scalar coupling constant** (NMR interaction strength)
between two atoms in a molecule. Mini version of Kaggle's *CHAMPS Predicting Molecular
Properties*. `structures.csv` gives each atom's element and 3-D position.

- **Task:** regression per atom pair; the `type` column (1JHC, 2JHH, ...) encodes the
  coupling type and dominates the target's scale
- **Metric:** (per the original comp) log of MAE, averaged over coupling types —
  lower is better
- **Kaggle link:** _TODO: add link_

**Approach:** join the 3-D coordinates of both atoms, compute the **distance** between
them (the single most important feature), add element/type information, and train one
LightGBM model.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
structures = pd.read_csv(f"{DATA_DIR}/structures.csv")
print(train.shape, test.shape, structures.shape)

(30000, 6) (8000, 5) (473049, 6)


In [2]:
def add_features(df):
    for i in (0, 1):
        df = df.merge(structures.add_suffix(f"_{i}"),
                      left_on=["molecule_name", f"atom_index_{i}"],
                      right_on=[f"molecule_name_{i}", f"atom_index_{i}"], how="left")
    d = df[["x_0","y_0","z_0"]].values - df[["x_1","y_1","z_1"]].values
    df["dist"] = np.linalg.norm(d, axis=1)
    df["dist_inv2"] = 1 / df["dist"] ** 2
    df["n_bonds"] = df["type"].str[0].astype(int)          # 1J / 2J / 3J
    df["type_code"] = df["type"].astype("category").cat.codes
    df["atom1_code"] = df["atom_1"].astype("category").cat.codes  # atom_0 is always H
    for c in ["atomic_number_1", "mulliken_charge_0", "mulliken_charge_1"]:
        if c not in df: df[c] = 0
    return df

train = add_features(train)
test  = add_features(test)
features = ["dist", "dist_inv2", "n_bonds", "type_code", "atom1_code",
            "atomic_number_1", "mulliken_charge_0", "mulliken_charge_1"]
features = [f for f in features if f in train.columns and f in test.columns]
print(features)

['dist', 'dist_inv2', 'n_bonds', 'type_code', 'atom1_code', 'atomic_number_1', 'mulliken_charge_0', 'mulliken_charge_1']


In [3]:
# Competition metric: mean over coupling types of log(MAE of that type)
def group_log_mae(y_true, y_pred, types):
    df = pd.DataFrame({"y": y_true, "p": y_pred, "t": types})
    maes = df.groupby("t").apply(lambda g: mean_absolute_error(g.y, g.p), include_groups=False)
    return np.log(maes.clip(lower=1e-9)).mean()

X, y = train[features], train["scalar_coupling_constant"].values
oof = np.zeros(len(train)); preds = np.zeros(len(test))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=0).split(X):
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.07, num_leaves=63,
                          random_state=0, verbose=-1)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict(X.iloc[va_idx])
    preds += m.predict(test[features]) / 5

print(f"CV MAE          : {mean_absolute_error(y, oof):.4f}")
print(f"CV group log-MAE: {group_log_mae(y, oof, train['type']):.4f}  (lower is better)")

CV MAE          : 2.7892
CV group log-MAE: 0.9262  (lower is better)


In [4]:
sub = pd.DataFrame({"id": test["id"], "scalar_coupling_constant": preds})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,scalar_coupling_constant
0,test_000000,0.712449
1,test_000001,88.478539
2,test_000002,114.971882
3,test_000003,-1.062816
4,test_000004,-10.872362


## Ideas to improve

- Train **one model per coupling type** — the 6 types behave very differently.
- Add neighborhood features: distances to the nearest atoms of each element,
  angles for 2J/3J paths, counts of atoms within a radius.
- The Kaggle winners used message-passing **graph neural networks** and transformer
  models over the molecular graph.
